In [1]:
!git clone https://github.com/reczoo/WWW2025_MMCTR_Challenge.git
%cd WWW2025_MMCTR_Challenge
!pip install -q fuxictr==2.3.10 polars "ray[data]>=2.52.0" pyarrow pydantic h5py modelscope

Cloning into 'WWW2025_MMCTR_Challenge'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 61 (delta 22), reused 52 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 68.80 KiB | 4.30 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/kaggle/working/WWW2025_MMCTR_Challenge


In [5]:
# dataset
!modelscope download --dataset reczoo/MicroLens_1M_x2 --local_dir /kaggle/working/tmp
!mkdir -p data/MicroLens_1M_x2
!unzip -q /kaggle/working/tmp/MicroLens_1M_x2.zip -d data/MicroLens_1M_x2
!ls data/MicroLens_1M_x2


 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'

2026-11-22 21:18:45,373 | INFO    | modelscope_hub.le

In [6]:
!unzip -q -o /kaggle/working/tmp/MicroLens_1M_x2.zip -d data/MicroLens_1M_x2
!ls -lh data/MicroLens_1M_x2

total 967M
-rw-r--r-- 1 root root  51M Oct 10 23:18 item_info.parquet
-rw-r--r-- 1 root root 591M Oct 16 15:42 item_raw_feature.parquet
-rw-r--r-- 1 root root  14K Nov  8 21:34 README
-rw-r--r-- 1 root root  44M Oct 11 09:13 test_unlabeled.parquet
-rw-r--r-- 1 root root 241M Oct 10 23:38 train.parquet
-rw-r--r-- 1 root root  42M Oct 10 23:38 valid.parquet


In [7]:
import torch
print(torch.__version__, torch.cuda.is_available())

2.11.0+cu130 True


In [8]:
cfg = "config/DIN_microlens_mmctr_tuner_config_01.yaml"
!sed -i 's/embedding_dim: \[32, 64\]/embedding_dim: [32]/' {cfg}
!sed -i 's/embedding_regularizer: \[1.e-6, 1.e-7\]/embedding_regularizer: [1.e-6]/' {cfg}
!tail -n 10 {cfg}


tuner_space:
    embedding_dim: [32]
    embedding_regularizer: [1.e-6]
    net_regularizer: 0
    net_dropout: 0.1
    dnn_hidden_units: [[1024, 512, 256]]
    din_use_softmax: true
    learning_rate: 1.e-3
    batch_size: 8192


In [10]:
# Entrainnement 
!python run_param_tuner.py --config config/DIN_microlens_mmctr_tuner_config_01.yaml --gpu 0

Enumerate all tuner configurations done.
2026-11-22 21:45:31,692 P229 INFO FuxiCTR version: 2.3.10
2026-11-22 21:45:31,692 P229 INFO Params: {
    "accumulation_steps": "1",
    "attention_dropout": "0.1",
    "attention_hidden_activations": "ReLU",
    "attention_hidden_units": "[512, 256]",
    "attention_output_activation": "None",
    "batch_norm": "True",
    "batch_size": "8192",
    "data_format": "parquet",
    "data_root": "./data/",
    "dataset_id": "MicroLens_1M_x2",
    "debug_mode": "False",
    "din_use_softmax": "True",
    "dnn_activations": "ReLU",
    "dnn_hidden_units": "[1024, 512, 256]",
    "early_stop_patience": "3",
    "embedding_dim": "32",
    "embedding_regularizer": "1e-06",
    "epochs": "100",
    "eval_steps": "None",
    "feature_cols": "[{'active': True, 'dtype': 'int', 'name': 'user_id', 'type': 'meta'}, {'active': True, 'dtype': 'int', 'name': 'hist_seq', 'type': 'meta'}, {'active': True, 'dtype': 'int', 'name': 'likes_level', 'type': 'categorical',

In [11]:
!cat DIN_microlens_mmctr_tuner_config_01.csv
!ls checkpoints/MicroLens_1M_x2
!ls config

 20261122-224328,[command] python run_expid.py --config config/DIN_microlens_mmctr_tuner_config_01 --expid DIN_MicroLens_1M_x2_001_bb2da5f7 --gpu 0,[exp_id] DIN_MicroLens_1M_x2_001_bb2da5f7,[dataset_id] MicroLens_1M_x2,[train] N.A.,[val] AUC: 0.759709 - logloss: 0.429133
DIN_MicroLens_1M_x2_001_bb2da5f7.log  DIN_MicroLens_1M_x2_001_bb2da5f7.model
din_config			     DIN_microlens_mmctr_tuner_config_01.yaml
DIN_microlens_mmctr_tuner_config_01


In [12]:
!python prediction.py --config config/DIN_microlens_mmctr_tuner_config_01 --expid DIN_MicroLens_1M_x2_001_bb2da5f7 --gpu 0
!ls -lh submission

2026-11-22 22:56:49,094 P446 INFO FuxiCTR version: 2.3.10
2026-11-22 22:56:49,094 P446 INFO Params: {
    "accumulation_steps": "1",
    "attention_dropout": "0.1",
    "attention_hidden_activations": "ReLU",
    "attention_hidden_units": "[512, 256]",
    "attention_output_activation": "None",
    "batch_norm": "True",
    "batch_size": "8192",
    "data_format": "parquet",
    "data_root": "./data/",
    "dataset_id": "MicroLens_1M_x2",
    "debug_mode": "False",
    "din_use_softmax": "True",
    "dnn_activations": "ReLU",
    "dnn_hidden_units": "[1024, 512, 256]",
    "early_stop_patience": "3",
    "embedding_dim": "32",
    "embedding_regularizer": "1e-06",
    "epochs": "100",
    "eval_steps": "None",
    "feature_cols": "[{'active': True, 'dtype': 'int', 'name': 'user_id', 'type': 'meta'}, {'active': True, 'dtype': 'int', 'name': 'hist_seq', 'type': 'meta'}, {'active': True, 'dtype': 'int', 'name': 'likes_level', 'type': 'categorical', 'vocab_size': 11}, {'active': True, 'dty